In [7]:
# 1. IMPORT
import pandas as pd
import re
from tqdm import tqdm
from collections import Counter
from underthesea import word_tokenize
from gensim.models import FastText

In [8]:
# 2. LOAD DATA
file_path = "C:/Users/user/OneDrive/Desktop/Do_an_co_so/DACS/data/TONG_HOP.xlsx"
df = pd.read_excel(file_path)

texts = df['sentence'].astype(str).tolist()


In [9]:
# 3. TIỀN XỬ LÝ
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)       # remove link
    text = re.sub(r"[^a-zA-ZÀ-ỹ\s]", " ", text)  # remove special char & number
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned_texts = [clean_text(t) for t in texts]


In [10]:
# 4. TOKENIZE
tokenized_texts = []
for text in tqdm(cleaned_texts):
    tokens = word_tokenize(text)
    tokenized_texts.append(tokens)

100%|██████████| 28093/28093 [00:23<00:00, 1220.14it/s]


In [11]:
# 6. RULE-BASED FILTER TEENCODE
def is_teencode(word):
    # 🔥 đưa dict vào trong hàm (fix tuyệt đối)
    vietnamese_dict = set([
        "tôi","bạn","không","có","là","đi","được","này","rồi","và",
        "ở","trong","khi","với","cho","về","rất","thì","mình",
        "các","đã","đang","sẽ","như","đó","đây","kia","vậy","nên"
    ])

    if len(word) <= 1:
        return False

    if word.isdigit():
        return False

    if word in vietnamese_dict:
        return False

    # lặp ký tự
    if re.search(r"(.)\1{2,}", word):
        return True

    # viết tắt
    if len(word) <= 3:
        return True

    # ký tự lạ
    if not re.match(r"^[a-zà-ỹ]+$", word):
        return True

    return False

In [12]:
# 7. TRÍCH XUẤT CANDIDATE
teencode_candidates = []

for tokens in tokenized_texts:
    for w in tokens:
        if is_teencode(w):
            teencode_candidates.append(w)


In [13]:
# 8. LỌC BẰNG FREQUENCY
counter = Counter(teencode_candidates)

# chỉ giữ từ xuất hiện >= 5 lần
teencode_filtered = [w for w, c in counter.items() if c >= 5]

print("Số teencode:", len(teencode_filtered))


Số teencode: 2654


In [14]:
# 9. TRAIN FASTTEXT
model = FastText(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4
)

In [15]:
# 10. MỞ RỘNG TEENCODE BẰNG FASTTEXT
expanded_teencode = set(teencode_filtered)

for word in teencode_filtered:
    try:
        similar_words = model.wv.most_similar(word, topn=5)
        for sim_word, score in similar_words:
            if score > 0.7:
                expanded_teencode.add(sim_word)
    except:
        continue

In [16]:
# 11. LÀM SẠCH FINAL LIST
final_teencode = sorted([
    w for w in expanded_teencode
    if len(w) > 1 and not w.isdigit()
])

print("Final teencode:", len(final_teencode))

Final teencode: 4492


In [17]:
# 12. SAVE FILE EXCEL
import pandas as pd

# Chuyển list thành DataFrame
df_teencode = pd.DataFrame(final_teencode, columns=["teencode"])

# Lưu thành file Excel
df_teencode.to_excel("teencode_list.xlsx", index=False)

print("Đã lưu file teencode_list.xlsx")

Đã lưu file teencode_list.xlsx
